# 01 — Train the Plagiarism Similarity Model (PyTorch fine-tune of Sentence-BERT)

> **Note on which corpus was actually used.** The checkpoint that ships with the
> backend was produced by `train.py` on the **Quora Question Pairs** corpus (30,000
> labelled pairs), and Chapter 5 of the report evaluates that checkpoint. This
> notebook is the earlier MIT-dataset route from the proposal, kept because it
> documents the approach and still works: `train.py --dataset csv --data <file>`
> trains on the same MIT CSV from the command line.

We take a *pretrained* `all-MiniLM-L6-v2` and **fine-tune** it on labelled sentence
pairs. Fine-tuning **is** training: it produces a new checkpoint we own.

- **Framework:** PyTorch (via `sentence-transformers`, which is pure PyTorch under the hood).
- **Input:** sentence pairs -> label (1 = plagiarised/duplicate, 0 = original).
- **Output:** a saved checkpoint the Django backend auto-loads.

### Dataset for this notebook
The **MIT Plagiarism Detection Dataset** (366,915 labelled sentence pairs).
Download from Kaggle and unzip into `notebooks/data/`:
https://www.kaggle.com/datasets/ruvelpereira/mit-plagairism-detection-dataset

Expected file: `data/train_snli.txt` or a CSV with two text columns + a label column.
The loader below is tolerant of column names.


In [ ]:
# 1. Install dependencies (run once)
# !pip install torch sentence-transformers datasets pandas scikit-learn
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

In [ ]:
# 2. Load the dataset (tolerant loader)
import os
import pandas as pd

DATA_DIR = 'data'

def load_pairs(data_dir=DATA_DIR):
    """Load sentence pairs + label from whatever the MIT dataset file is called.
    Returns a DataFrame with columns: text_a, text_b, label (int 0/1)."""
    candidates = [f for f in os.listdir(data_dir)
                  if f.endswith(('.csv', '.txt', '.tsv'))]
    if not candidates:
        raise FileNotFoundError(
            f'No data file in {data_dir}/. Download the MIT Plagiarism dataset from '
            'https://www.kaggle.com/datasets/ruvelpereira/mit-plagairism-detection-dataset')
    path = os.path.join(data_dir, candidates[0])
    print('Loading', path)
    sep = '\t' if path.endswith(('.txt', '.tsv')) else ','
    df = pd.read_csv(path, sep=sep)
    # Normalise column names: find the two text cols and the label col.
    cols = [c.lower() for c in df.columns]
    df.columns = cols
    text_cols = [c for c in cols if any(k in c for k in ('sentence', 'text', 'source', 'target', 'question'))]
    label_col = next((c for c in cols if any(k in c for k in ('label', 'plag', 'similar', 'is_dup'))), cols[-1])
    a, b = text_cols[0], text_cols[1]
    out = df[[a, b, label_col]].copy()
    out.columns = ['text_a', 'text_b', 'label']
    out = out.dropna()
    out['label'] = out['label'].astype(int)
    return out

df = load_pairs()
print(df.shape)
df.head()

In [ ]:
# 3. Explore + subsample (subsample keeps the MVP run fast)
print('Label balance:\n', df['label'].value_counts())

SAMPLE_SIZE = 30000   # bump to None to use the full 366k for the final model
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.groupby('label', group_keys=False).apply(
        lambda g: g.sample(min(len(g), SAMPLE_SIZE // 2), random_state=42))
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print('Using', len(df), 'pairs')

In [ ]:
# 4. Train / test split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
print('train:', len(train_df), ' test:', len(test_df))

In [ ]:
# 5. Build the model + training data (Sentence-BERT / siamese network, PyTorch)
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

# CosineSimilarityLoss treats the label as the TARGET cosine similarity (0.0 or 1.0).
# The siamese network learns to place plagiarised pairs close, originals far apart.
train_examples = [
    InputExample(texts=[r.text_a, r.text_b], label=float(r.label))
    for r in train_df.itertuples(index=False)
]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = losses.CosineSimilarityLoss(model)
print('Training examples:', len(train_examples))

In [ ]:
# 6. Fine-tune (this is the training step) -- a few minutes on GPU, ~20-30 min CPU
EPOCHS = 1                     # 1 epoch is enough for an MVP; try 2-3 for the final model
warmup = int(len(train_loader) * 0.1)

model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    show_progress_bar=True,
)
print('Done training.')

In [ ]:
# 7. Evaluate: cosine similarity + threshold -> accuracy / F1
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

emb_a = model.encode(test_df['text_a'].tolist(), convert_to_numpy=True, show_progress_bar=True)
emb_b = model.encode(test_df['text_b'].tolist(), convert_to_numpy=True, show_progress_bar=True)
cos = (emb_a * emb_b).sum(1) / (np.linalg.norm(emb_a, axis=1) * np.linalg.norm(emb_b, axis=1) + 1e-8)

y_true = test_df['label'].values
THRESHOLD = 0.7               # same default the backend uses
y_pred = (cos >= THRESHOLD).astype(int)
print(f'Accuracy: {accuracy_score(y_true, y_pred):.3f}')
print(f'F1:       {f1_score(y_true, y_pred):.3f}')
print(f'ROC-AUC:  {roc_auc_score(y_true, cos):.3f}')

In [ ]:
# 8. Save the fine-tuned checkpoint where the Django backend expects it
SAVE_PATH = '../backend/detector/models/plagiarism-sbert'
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
model.save(SAVE_PATH)
print('Saved fine-tuned model ->', SAVE_PATH)
print('The backend loads this automatically if it exists.')

In [ ]:
# 9. Quick sanity demo
pairs = [
    ('The cat sat on the mat.', 'A cat was sitting on the mat.'),        # paraphrase -> high
    ('The economy grew last year.', 'I love eating pizza on Sundays.'),  # unrelated  -> low
]
for a, b in pairs:
    ea, eb = model.encode([a, b])
    sim = float((ea @ eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))
    print(f'{sim:.3f}  |  {a!r}  <>  {b!r}')

## Appendix — the same idea as a from-scratch PyTorch loop (for your defense)

`model.fit()` above hides the training loop. The cell below shows what it does under the
hood so you can explain forward pass / loss / backward pass / optimizer step. It is a
*minimal* illustration on a tiny batch — not meant to replace the real training above.

In [ ]:
# Minimal manual training step (illustration only)
import torch
import torch.nn.functional as F

opt = torch.optim.AdamW(model.parameters(), lr=2e-5)

batch = train_df.head(8)
labels = torch.tensor(batch['label'].values, dtype=torch.float, device=DEVICE)

# forward pass: encode both sides, cosine similarity
feat_a = model.tokenize(batch['text_a'].tolist())
feat_b = model.tokenize(batch['text_b'].tolist())
feat_a = {k: v.to(DEVICE) for k, v in feat_a.items()}
feat_b = {k: v.to(DEVICE) for k, v in feat_b.items()}
emb_a = model(feat_a)['sentence_embedding']
emb_b = model(feat_b)['sentence_embedding']
cos = F.cosine_similarity(emb_a, emb_b)

# loss: predicted cosine should match the 0/1 label (MSE)
loss = F.mse_loss(cos, labels)
print('loss before step:', float(loss))

# backward pass + optimizer step
opt.zero_grad()
loss.backward()
opt.step()
print('One manual gradient step done.')